# 01 – Probability Basics

Statistics and machine learning are built on probability.

When a model says a customer is "70% likely to churn," that's a probability statement. When a doctor says a test is "95% accurate," that's probability. When you're told a stock has a "1 in 20 chance of a 10% drop," that's probability.

Before you can understand hypothesis testing, distributions, or model evaluation, you need a firm grip on the basics.

This notebook covers:
- What probability is, and what it measures
- The rules that govern it
- Conditional probability — the most important concept for data science
- Bayes' theorem — how to update beliefs with evidence

## 1) What is Probability?

Probability measures **how likely an event is to happen**, on a scale from 0 to 1.

- `P = 0` → impossible (will never happen)
- `P = 1` → certain (will always happen)
- `P = 0.5` → equally likely to happen or not happen

```
P(event) = (number of ways the event can happen) / (total number of equally likely outcomes)
```

**Example:** A fair six-sided die. What's the probability of rolling a 4?

- Ways to get a 4: 1 (only one face shows 4)
- Total outcomes: 6

`P(rolling 4) = 1/6 ≈ 0.167`

In [ ]:
import numpy as np
import pandas as pd

# Simulate rolling a die 60,000 times — law of large numbers in action
rng = np.random.default_rng(42)
rolls = rng.integers(1, 7, size=60_000)

print("Die Roll Probabilities (simulated vs theoretical):")
print("-" * 45)
for face in range(1, 7):
    simulated  = (rolls == face).mean()
    theoretical = 1 / 6
    print(f"  Face {face}: simulated={simulated:.4f}  theoretical={theoretical:.4f}  diff={abs(simulated-theoretical):.4f}")

With 60,000 rolls, the simulated probabilities are very close to the theoretical 1/6 ≈ 0.1667.

This is the **Law of Large Numbers**: as you repeat an experiment more times, the observed frequencies converge to the true probabilities. With only 10 rolls, you'd see much larger differences from 1/6.

## 2) Complementary Events

If P(A) is the probability that event A happens, then:

```
P(not A) = 1 - P(A)
```

This is one of the most useful rules because it lets you flip the question.

**Example:** If the probability of rain tomorrow is 0.3, the probability of no rain is 0.7.

`P(no rain) = 1 - 0.3 = 0.7`

In [ ]:
# Exam pass/fail example
# Historical data: 72% of students pass a competitive exam

p_pass = 0.72
p_fail = 1 - p_pass

print(f"P(pass)         = {p_pass:.2f}")
print(f"P(fail)         = {p_fail:.2f}")
print(f"Sum (must = 1)  = {p_pass + p_fail:.2f}")
print()

# More useful: at least one success in multiple independent attempts
p_pass_one_attempt  = 0.40
p_fail_one_attempt  = 1 - p_pass_one_attempt

# P(at least one pass in 3 attempts) = 1 - P(fail all 3)
p_fail_all_3        = p_fail_one_attempt ** 3
p_at_least_one_pass = 1 - p_fail_all_3

print(f"P(pass on one attempt)   = {p_pass_one_attempt:.2f}")
print(f"P(fail all 3 attempts)   = {p_fail_all_3:.4f}")
print(f"P(pass at least once)    = {p_at_least_one_pass:.4f}")

The complementary approach is often easier than computing the direct probability.

"At least one pass in 3 attempts" could mean: pass on 1st, or 2nd, or 3rd, or 1st and 2nd, or... — many cases to count.

The complement — "fail all 3" — is just one case, easy to compute.

## 3) AND and OR — Combining Events

### AND (Intersection) — Both events happen

For **independent** events (one doesn't affect the other):

```
P(A and B) = P(A) × P(B)
```

**Example:** What's the probability of getting heads twice in two coin flips?

`P(heads AND heads) = 0.5 × 0.5 = 0.25`

### OR (Union) — At least one event happens

```
P(A or B) = P(A) + P(B) - P(A and B)
```

Why subtract? Because when we add P(A) and P(B), we count the overlap (both happening) twice.

**Special case — mutually exclusive events** (can't both happen):
If A and B can't happen at the same time, `P(A and B) = 0`, so:

```
P(A or B) = P(A) + P(B)
```

In [ ]:
# AND example: both events happening
p_sunny     = 0.60   # P(sunny day)
p_match_win = 0.55   # P(team wins their match)

# Independent events
p_both = p_sunny * p_match_win
print(f"P(sunny)              = {p_sunny:.2f}")
print(f"P(team wins)          = {p_match_win:.2f}")
print(f"P(sunny AND team wins)= {p_both:.4f}")
print()

# OR example: at least one event happening
p_a = 0.30   # P(event A)
p_b = 0.25   # P(event B)
p_ab = p_a * p_b  # assuming independent

p_a_or_b = p_a + p_b - p_ab
print(f"P(A)        = {p_a:.2f}")
print(f"P(B)        = {p_b:.2f}")
print(f"P(A AND B)  = {p_ab:.4f}")
print(f"P(A OR B)   = {p_a_or_b:.4f}")
print()

# Mutually exclusive: rolling a 2 OR a 5 on a die
p_two = 1/6
p_five = 1/6
# Can't roll both at once — mutually exclusive
p_two_or_five = p_two + p_five
print(f"P(rolling 2 OR 5) = {p_two_or_five:.4f}  (= 2/6)")

**When events are mutually exclusive:** Getting heads and tails on the same flip — impossible. Rolling a 2 and a 5 on the same roll — impossible.

**When events are not mutually exclusive:** Being in Pune and being a Science student — both can be true at the same time.

The general formula `P(A or B) = P(A) + P(B) - P(A and B)` always works. The mutually exclusive version is just a special case where the subtracted term is zero.

## 4) Conditional Probability — The Most Important Concept

**Conditional probability** answers: *Given that we already know something, how does that change our probability estimate?*

```
P(A | B) = P(A and B) / P(B)
```

Read as: "Probability of A, **given** B has happened."

**Example:**

In a class of 100 students:
- 40 study Science
- 25 are in Science AND scored above 80
- What's the probability that a student scored above 80, **given** they're in Science?

```
P(above 80 | Science) = P(above 80 AND Science) / P(Science)
                      = 25/100 ÷ 40/100
                      = 25/40
                      = 0.625
```

62.5% of Science students scored above 80 — but only 25% of all students did.
Knowing someone is in Science changed your estimate dramatically.

In [ ]:
# Student dataset example
rng = np.random.default_rng(42)
n = 1000

students = pd.DataFrame({
    "Department": rng.choice(["Science","Arts","Commerce"], size=n, p=[0.40,0.30,0.30]),
    "Score":      np.clip(rng.normal(68, 18, n).round(0).astype(int), 0, 100)
})

total = len(students)
science_students  = (students["Department"] == "Science").sum()
high_scorers      = (students["Score"] > 80).sum()
science_and_high  = ((students["Department"] == "Science") & (students["Score"] > 80)).sum()

p_science         = science_students / total
p_high            = high_scorers     / total
p_science_and_high= science_and_high / total

# Conditional probability: P(high scorer | Science)
p_high_given_science = p_science_and_high / p_science

print(f"Total students      : {total}")
print(f"In Science          : {science_students}  → P(Science) = {p_science:.3f}")
print(f"Score > 80          : {high_scorers}   → P(High)    = {p_high:.3f}")
print(f"Science AND high    : {science_and_high}")
print()
print(f"P(High | Science)   = {p_high_given_science:.3f}")
print(f"P(High) overall     = {p_high:.3f}")
print()
print(f"Science background increases the chance of a high score: {p_high_given_science > p_high}")

When `P(A|B) ≠ P(A)`, the events are **dependent** — knowing B happened changes your estimate of A.

When `P(A|B) = P(A)`, the events are **independent** — knowing B tells you nothing new about A.

This is fundamental to feature selection in machine learning: if knowing a feature's value changes your prediction of the label, the feature is informative. If it doesn't, the feature is useless.

## 5) Bayes' Theorem — Updating Beliefs with Evidence

Bayes' theorem lets you **reverse a conditional probability**.

If you know P(B|A), you can find P(A|B):

```
P(A | B) = P(B | A) × P(A) / P(B)
```

In plain English:

```
P(hypothesis | evidence) = P(evidence | hypothesis) × P(hypothesis) / P(evidence)
```

- **P(A)** → your belief before seeing any evidence (**prior**)
- **P(A|B)** → your updated belief after seeing evidence B (**posterior**)
- **P(B|A)** → how likely is B if A is true (**likelihood**)
- **P(B)** → how common is B overall (**normalising constant**)

### The Classic Medical Test Example

A disease affects 1% of the population. A test for the disease is:
- **95% sensitive** — if you have the disease, it correctly returns positive 95% of the time
- **90% specific** — if you don't have the disease, it correctly returns negative 90% of the time (so 10% false positive rate)

You test positive. What's the probability you actually have the disease?

Most people's gut answer: "95%." The actual answer is much lower — and Bayes' theorem shows you why.

In [ ]:
# Bayes' theorem — medical test example

p_disease          = 0.01   # 1% of population has the disease (prior)
p_no_disease       = 1 - p_disease

p_pos_given_sick   = 0.95   # sensitivity (true positive rate)
p_pos_given_healthy= 0.10   # false positive rate (1 - specificity)

# P(positive) — how common is a positive test overall?
# = P(positive | sick) × P(sick) + P(positive | healthy) × P(healthy)
p_positive = p_pos_given_sick * p_disease + p_pos_given_healthy * p_no_disease

# Bayes' theorem: P(sick | positive)
p_sick_given_positive = (p_pos_given_sick * p_disease) / p_positive

print("=== Medical Test Scenario ===")
print(f"Disease prevalence       : {p_disease:.0%}")
print(f"Test sensitivity         : {p_pos_given_sick:.0%}  (true positive rate)")
print(f"False positive rate      : {p_pos_given_healthy:.0%}")
print()
print(f"P(positive test overall) : {p_positive:.4f}")
print()
print(f"P(actually sick | positive test) = {p_sick_given_positive:.3f}  ({p_sick_given_positive:.1%})")
print()
print("Intuition: Despite a positive test, there's only a 8.7% chance of actually having")
print("the disease — because the disease is rare (1%) relative to false positives (10%).")

This result shocks most people. The math is correct.

Why? In 10,000 people:
- 100 have the disease. Of those, 95 test positive (true positives).
- 9,900 are healthy. Of those, 990 test positive (false positives).
- Total positive tests: 95 + 990 = 1,085
- Of those, only 95 are real: 95 / 1,085 ≈ **8.75%**

The low prevalence (1%) drowns out the test's accuracy.

This matters for any rare-event detection: fraud, disease, equipment failure. A model with 95% accuracy can still be mostly wrong when the positive class is very rare.

In [ ]:
# Show how prevalence changes the posterior
import pandas as pd

prevalences = [0.001, 0.01, 0.05, 0.10, 0.20, 0.50]
sensitivity = 0.95
fpr         = 0.10

rows = []
for prev in prevalences:
    p_pos = sensitivity * prev + fpr * (1 - prev)
    posterior = (sensitivity * prev) / p_pos
    rows.append({
        "Prevalence": f"{prev:.1%}",
        "P(sick | positive test)": f"{posterior:.1%}"
    })

print("How prevalence affects the value of a positive test:")
print("(Sensitivity = 95%, False Positive Rate = 10%)")
print()
print(pd.DataFrame(rows).to_string(index=False))

The posterior probability of being sick — even with a positive test — ranges from 1% (very rare disease) to 91% (disease affects 50% of people).

The same test, the same result, wildly different meanings depending on the base rate.

This is one of the most practical statistical insights: **always consider the base rate before interpreting a classification result.**

## 6) Putting It Together — Expected Value

**Expected value** is the average outcome you'd get if you repeated a random experiment infinitely many times.

```
E[X] = Σ (outcome × probability of that outcome)
```

In [ ]:
# Expected value of a die roll
outcomes      = [1, 2, 3, 4, 5, 6]
probabilities = [1/6] * 6

expected_value = sum(o * p for o, p in zip(outcomes, probabilities))
print(f"Expected value of a die roll: {expected_value:.4f}")
print(f"(Check: (1+2+3+4+5+6)/6 = {sum(outcomes)/6})")
print()

# Business example: insurance
# A policy costs ₹5,000/year
# Events and payouts:
events = [
    ("Minor claim",  0.10, 15_000),
    ("Major claim",  0.02, 150_000),
    ("No claim",     0.88, 0),
]

print("Insurance Expected Payout:")
total_expected_payout = 0
for name, prob, payout in events:
    contribution = prob * payout
    total_expected_payout += contribution
    print(f"  {name:<15}: P={prob:.2f}  Payout=₹{payout:>8,}  Contribution=₹{contribution:>8,.0f}")

print(f"Expected payout per customer = ₹{total_expected_payout:,.0f}")
print(f"Premium collected           = ₹{5000:,.0f}")
print(f"Expected profit per customer= ₹{5000 - total_expected_payout:,.0f}")

Insurance companies, casinos, and financial institutions live by expected value. They don't care what happens on any single transaction — they care about the average over millions of transactions.

For a data scientist, expected value shows up in: cost-benefit analysis of model decisions, evaluating classifiers with asymmetric misclassification costs, and designing experiments.

## Summary

| Concept | Formula | When you use it |
|---|---|---|
| Basic probability | `P(A) = favourable / total` | Foundational |
| Complement | `P(not A) = 1 - P(A)` | "At least one" problems |
| AND (independent) | `P(A and B) = P(A) × P(B)` | Joint probability |
| OR (general) | `P(A or B) = P(A) + P(B) - P(A and B)` | Either/or |
| Conditional | `P(A\|B) = P(A and B) / P(B)` | Given information |
| Bayes | `P(A\|B) = P(B\|A) × P(A) / P(B)` | Updating beliefs |
| Expected value | `E[X] = Σ outcome × P` | Decision making |

**Key ideas to carry forward:**
- Rare events produce many false positives even with accurate tests — always check the base rate
- Conditional probability is the core of all predictive modelling
- Bayes' theorem is how you rationally update beliefs with evidence

Next up: **Random Variables** — describing uncertain quantities mathematically.

## Practice Questions 1

1. A bag has 4 red and 6 blue marbles. You draw one marble without looking.
   - What is P(red)?
   - What is P(not red)?
   - If you draw two marbles **with replacement**, what is P(both red)?

2. In a call centre, 60% of calls are resolved on the first attempt. What is the probability that at least one of the next 3 calls requires a second attempt? (Use the complement method.)

3. In a dataset: 55% of customers are from Mumbai, 30% have premium accounts, and 20% are from Mumbai AND have premium accounts. What is P(Mumbai OR premium)?

## Practice Questions 2

1. A factory has two machines (A and B). Machine A produces 60% of the output. Of Machine A's output, 3% is defective. Of Machine B's output, 5% is defective. A randomly selected item is defective. What is the probability it came from Machine A? (Use Bayes' theorem.)

2. A spam filter flags emails as spam with:
   - True positive rate (spam correctly flagged): 92%
   - False positive rate (ham incorrectly flagged): 2%
   - 15% of all emails are spam
   An email is flagged as spam. What is the probability it is actually spam?

3. A game show: you win ₹10,000 with probability 0.1, ₹1,000 with probability 0.3, and lose ₹500 with probability 0.6. What is the expected value of playing?